In [1]:
# Практика 12. Довірчі інтервали для середнього і частки
# Коць Артем, ІТ-42
# Варіант 9 — Суми

import numpy as np
import pandas as pd
from scipy import stats

print("Місто: Суми")
print("Варіант: 9")
print("Липнева температура: 20 °C")

Місто: Суми
Варіант: 9
Липнева температура: 20 °C


In [2]:
np.random.seed(9)

daily_temps = np.random.normal(
    loc=20,
    scale=2.5,
    size=30
)

print("Вибірка денних температур:")
print(daily_temps)

print("\nСереднє:", daily_temps.mean())
print("Стандартне відхилення:", daily_temps.std(ddof=1))

Вибірка денних температур:
[20.00277139 19.27613983 17.20983424 19.96779311 19.05409634 18.79716159
 16.20667205 18.77282005 19.39829855 18.38013135 21.5897277  24.35029326
 20.74170554 21.76875915 24.55703941 21.07692257 23.85682406 17.74819707
 19.65718747 23.24394753 21.68817792 20.07989529 22.29536474 20.95127367
 21.29091872 19.11190135 20.5219425  20.82102769 18.75443808 14.77055807]

Середнє: 20.198060677209607
Стандартне відхилення: 2.253961543383256


In [3]:
n = len(daily_temps)
mean = daily_temps.mean()
s = daily_temps.std(ddof=1)

se = s / np.sqrt(n)

ci_mean = stats.t.interval(
    0.95,
    df=n - 1,
    loc=mean,
    scale=se
)

print("n =", n)
print("Середнє =", mean)
print("Вибіркове стандартне відхилення =", s)
print("Стандартна похибка =", se)
print("95% довірчий інтервал для середнього:", ci_mean)

n = 30
Середнє = 20.198060677209607
Вибіркове стандартне відхилення = 2.253961543383256
Стандартна похибка = 0.4115151936867228
95% довірчий інтервал для середнього: (np.float64(19.35641760489354), np.float64(21.039703749525675))


### Завдання 2. Довірчий інтервал для середньої температури

Для вибірки з 30 денних температур обчислено вибіркове середнє,
стандартне відхилення та стандартну похибку:

SE = s / √n.

95% довірчий інтервал побудовано за допомогою t-розподілу через
`stats.t.interval()`. t-розподіл використовується тому, що справжнє
стандартне відхилення генеральної сукупності σ невідоме, а замість
нього використовується вибіркове стандартне відхилення s.

Використовувати звичайне z-критичне значення тут недоцільно, оскільки
σ невідоме. Для вибірки такого розміру використання t-розподілу
враховує додаткову невизначеність оцінювання σ за самою вибіркою.

In [4]:
p_hat = (daily_temps > 20).mean()

n_p = n * p_hat
n_1p = n * (1 - p_hat)

print("Частка днів із температурою понад 20 °C:", p_hat)
print("n * p_hat =", n_p)
print("n * (1 - p_hat) =", n_1p)

Частка днів із температурою понад 20 °C: 0.5333333333333333
n * p_hat = 16.0
n * (1 - p_hat) = 14.0


In [5]:
se_p = np.sqrt(p_hat * (1 - p_hat) / n)

ci_prop = stats.norm.interval(
    0.95,
    loc=p_hat,
    scale=se_p
)

print("Стандартна похибка частки =", se_p)
print("95% довірчий інтервал для частки:", ci_prop)
print("Чи входить 0.5 в інтервал?", ci_prop[0] <= 0.5 <= ci_prop[1])

Стандартна похибка частки = 0.09108400680852977
95% довірчий інтервал для частки: (np.float64(0.3548119604210139), np.float64(0.7118547062456527))
Чи входить 0.5 в інтервал? True


### Завдання 3. Довірчий інтервал для частки

Частка днів, коли температура перевищила липневе табличне значення
20 °C, обчислена як `p_hat`.

Для перевірки нормального наближення обчислено `n * p_hat` та
`n * (1 - p_hat)`. Обидва значення повинні бути достатньо великими
(орієнтовно не менше 5), тому використання нормального наближення
є прийнятним, якщо ця умова виконується для отриманої вибірки.

95% довірчий інтервал для частки побудовано за допомогою
`stats.norm.interval()`.

Теоретично частка днів із температурою вище 20 °C становить близько
0.5, оскільки вибірка симетрично згенерована навколо 20 °C.

Якщо 0.5 не потрапляє до отриманого інтервалу, це не означає, що
метод неправильний. Одна конкретна вибірка з 30 спостережень може
дати такий результат через випадковість. При повторенні процедури
побудови 95% довірчих інтервалів приблизно 95% таких інтервалів
накриватимуть істинне значення, а близько 5% — ні.

In [6]:
confidence_levels = [0.90, 0.95, 0.99]

results = []

for confidence in confidence_levels:
    interval = stats.t.interval(
        confidence,
        df=n - 1,
        loc=mean,
        scale=se
    )
    
    width = interval[1] - interval[0]
    
    results.append({
        "Рівень довіри": confidence,
        "Нижня межа": interval[0],
        "Верхня межа": interval[1],
        "Ширина інтервалу": width
    })

ci_table = pd.DataFrame(results)

ci_table

,Рівень довіри,Нижня межа,Верхня межа,Ширина інтервалу
0,0.90,19.498844,20.897277,1.398433
1,0.95,19.356418,21.039704,1.683286
2,0.99,19.063766,21.332355,2.268589


### Завдання 4. Ширина інтервалу при різних рівнях довіри

Для однієї й тієї самої вибірки побудовано довірчі інтервали
для рівнів довіри 90%, 95% і 99%.

Зі збільшенням рівня довіри ширина інтервалу також збільшується:
90% < 95% < 99%. Це відбувається тому, що для більшої впевненості
потрібно охопити ширший діапазон можливих значень середнього.

При незмінному рівні довіри вужчий інтервал можна отримати за рахунок
збільшення розміру вибірки n. Також ширина залежить від вибіркового
стандартного відхилення s: менша варіативність даних дає вужчий
інтервал.

Отже, основними факторами ширини довірчого інтервалу є рівень довіри,
варіативність даних і розмір вибірки.

### Завдання 5. Інтерпретація результату словами

Отриманий 95% довірчий інтервал для середньої липневої температури
міста Суми показує діапазон значень, отриманий за цією статистичною
процедурою.

Правильна частотна інтерпретація полягає в тому, що якби ми багато
разів генерували нові вибірки по 30 днів і щоразу будували 95%
довірчий інтервал тим самим способом, приблизно 95% побудованих
інтервалів накривали б істинне середнє значення температури.

Не можна говорити, що після побудови конкретного інтервалу існує
95% ймовірності того, що істинне середнє знаходиться саме в ньому.

## Контрольні питання

### 1. Яка різниця між точковою оцінкою і довірчим інтервалом?

Точкова оцінка — це одне число, наприклад вибіркове середнє x̄,
яке використовується як оцінка невідомого параметра генеральної
сукупності.

Довірчий інтервал дає діапазон можливих значень параметра та
враховує статистичну невизначеність вибіркової оцінки. Тому одного
числа недостатньо для повної оцінки, оскільки воно не показує
розмір похибки та невизначеність результату.

### 2. Чому використовують t-розподіл, а не звичайний нормальний?

Коли справжнє стандартне відхилення σ невідоме, його замінюють
вибірковим стандартним відхиленням s. За невеликої вибірки це додає
невизначеність, яку враховує t-розподіл.

Тому для довірчого інтервалу середнього в цій роботі використано
t-розподіл із кількістю ступенів свободи n - 1.

### 3. Що насправді означає «95% довіри»?

95% довіри означає властивість процедури побудови інтервалів:
якщо багато разів отримувати нові вибірки та будувати для кожної
95% довірчий інтервал однаковим методом, приблизно 95% цих
інтервалів накриватимуть істинне значення параметра.

Не можна говорити, що для вже побудованого конкретного інтервалу
ймовірність знаходження істинного значення всередині дорівнює 95%.
Істинне значення є фіксованим, а випадковим є сам інтервал до моменту
отримання вибірки.

### 4. Які три фактори визначають ширину довірчого інтервалу?

Ширину інтервалу визначають:

1. Рівень довіри — чим він вищий, тим ширший інтервал.
2. Варіативність даних — чим більше стандартне відхилення s,
   тим ширший інтервал.
3. Розмір вибірки n — чим більша вибірка, тим вужчий інтервал.

Для середнього це пов'язано зі стандартною похибкою
SE = s / √n. Тому збільшення вибірки зменшує невизначеність оцінки.